In [ ]:
import logging
from datetime import datetime, UTC
import yfinance as yf
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from src.config import DATABASE_URL, MARKET_TICKERS
from src.data.models import Base, MacroData

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

def fetch_market_data(ticker: str, start_date: str = "1950-01-01") -> yf.Ticker:
    """Télécharge l'historique quotidien d'un actif depuis Yahoo Finance."""
    logger.info(f"Téléchargement des données pour le ticker : {ticker}")
    tk = yf.Ticker(ticker)
    df = tk.history(start=start_date, interval="1d")
    return df

def save_market_to_sqlite(df, series_id: str) -> None:
    """Valide et stocke les cours de clôture dans la table Point-in-Time."""
    engine = create_engine(DATABASE_URL)
    Base.metadata.create_all(bind=engine)
    Session = sessionmaker(bind=engine)
    
    rows_added = 0
    
    with Session() as session:
        for idx, row in df.iterrows():
            # yfinance utilise la date comme index (idx)
            current_date = idx.date()
            close_price = float(row["Close"])
            
            # RÈGLE MARKET DATA : La date de publication est égale à la date du prix.
            # Un prix de clôture du jour J est disponible le jour J. Pas de révision future.
            macro_row = MacroData(
                series_id=series_id,
                source="YFINANCE",
                date=current_date,
                release_date=current_date,
                value=close_price,
                fetched_at=datetime.now(UTC)
            )
            
            session.merge(macro_row)
            rows_added += 1
            
        session.commit()
    
    logger.info(f"Stockage réussi pour {series_id} : {rows_added} lignes insérées/mises à jour.")

if __name__ == "__main__":
    logger.info("Démarrage de la pipeline d'ingestion Yahoo Finance.")
    
    for label, ticker in MARKET_TICKERS.items():
        try:
            df_data = fetch_market_data(ticker)
            if not df_data.empty:
                save_market_to_sqlite(df_data, label)
            else:
                logger.warning(f"Aucune donnée récupérée pour {label} ({ticker})")
        except Exception as error:
            logger.error(f"Erreur lors du traitement de l'actif {label} : {error}")